# Walmart M5 Data Understanding

## Business Objective

Before building the forecasting pipeline, it is important to understand the structure, quality, and relationships within the Walmart M5 dataset.

This notebook explores the source datasets, identifies potential data quality issues, and documents key characteristics that will guide database design, feature engineering, and forecasting.

## Import Libraries

The following libraries are used throughout this notebook for loading and inspecting the datasets.

In [15]:
import pandas as pd

## Load Datasets

The M5 competition provides five primary datasets containing calendar information, weekly prices, historical sales, evaluation data, and the competition submission structure.

Each dataset is loaded into a pandas DataFrame for initial inspection.

In [16]:
calendar = pd.read_csv("../data/raw/calendar.csv")
prices = pd.read_csv("../data/raw/sell_prices.csv")
sales = pd.read_csv("../data/raw/sales_train_validation.csv")
evaluation = pd.read_csv("../data/raw/sales_train_evaluation.csv")
submission = pd.read_csv("../data/raw/sample_submission.csv")

datasets = {
    "calendar": calendar,
    "prices": prices,
    "sales": sales,
    "evaluation": evaluation,
    "submission": submission,
}

## Dataset Overview

The first step is to understand the overall size of each dataset.

Knowing the number of rows and columns helps estimate storage requirements and computational complexity.

In [17]:
for name, df in datasets.items():
    print(f"{name.title()}:", df.shape)

Calendar: (1969, 14)
Prices: (6841121, 4)
Sales: (30490, 1919)
Evaluation: (30490, 1947)
Submission: (60980, 29)


### Observation

The datasets vary significantly in size. The sales datasets are much larger than the supporting datasets because they contain daily historical sales for thousands of product-store combinations.

## Preview the Dataset

Inspecting the first few records provides an initial understanding of the dataset's structure and column contents.

### Calendar Dataset

In [18]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


### Sell Prices Dataset

In [19]:
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


### Sales Validation Dataset

In [20]:
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


### Sales Evaluation Dataset

In [21]:
evaluation.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


### Sample Submission Dataset

In [22]:
submission.head()

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Observation

The dataset previews show how the M5 data is organized across several related files:

- The **calendar dataset** contains dates, Walmart week identifiers, events, and SNAP indicators.
- The **sell prices dataset** contains weekly product prices by store.
- The **sales validation dataset** contains product-level daily sales from `d_1` through `d_1913`.
- The **sales evaluation dataset** extends the historical sales period through `d_1941`.
- The **sample submission dataset** contains the `F1`–`F28` columns used to represent the 28-day forecasting horizon.

The sales data is stored in a wide format, with each day represented as a separate column. These datasets will later be connected and transformed into a structure suitable for analysis and demand forecasting.

## Examine Data Types

Understanding the data types helps identify which columns represent numerical values, categorical variables, dates, or identifiers.

## Data Types and Structure

Before analysis, the datasets are inspected to confirm their structure and identify the main data types used throughout the project.

In [23]:
for name, df in datasets.items():
    print(
        f"{name.upper():12} "
        f"Rows: {df.shape[0]:,} | "
        f"Columns: {df.shape[1]:,} | "
        f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
    )

CALENDAR     Rows: 1,969 | Columns: 14 | Memory: 0.26 MB
PRICES       Rows: 6,841,121 | Columns: 4 | Memory: 318.15 MB
SALES        Rows: 30,490 | Columns: 1,919 | Memory: 448.23 MB
EVALUATION   Rows: 30,490 | Columns: 1,947 | Memory: 454.74 MB
SUBMISSION   Rows: 60,980 | Columns: 29 | Memory: 15.16 MB


## Missing Value Analysis

Missing values are common in real-world datasets.

This section identifies which variables contain missing data and how frequently those missing values occur.

In [24]:

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.isnull().sum()[df.isnull().sum() > 0])


CALENDAR
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
dtype: int64

PRICES
Series([], dtype: int64)

SALES
Series([], dtype: int64)

EVALUATION
Series([], dtype: int64)

SUBMISSION
Series([], dtype: int64)


### Observation

Missing values occur only in the calendar event columns. This is expected because most dates do not correspond to holidays or special events. No unexpected missing values were identified in the remaining source datasets.

## Duplicate Record Analysis

Duplicate rows can negatively affect downstream analytics and machine learning models.

This section checks whether duplicate records exist.

In [25]:
for name, df in datasets.items():
    print(name, df.duplicated().sum())

calendar 0
prices 0
sales 0
evaluation 0
submission 0


### Observation

No duplicate rows were identified across any of the five datasets, indicating that the source files are internally consistent prior to ETL processing.

## Unique Value Analysis

Understanding the number of unique categories provides insight into the dimensionality of the dataset.

In [26]:
columns = {
    "Stores": "store_id",
    "Products": "item_id",
    "Departments": "dept_id",
    "Categories": "cat_id",
    "States": "state_id",
}

for name, column in columns.items():
    print(f"{name}:", sales[column].nunique())

Stores: 10
Products: 3049
Departments: 7
Categories: 3
States: 3


### Observation

The Walmart M5 dataset contains:

- 10 stores
- 3 states
- 3 product categories
- 7 departments
- 3,049 unique products

These identifiers will become important dimensions within the PostgreSQL warehouse.

### Observation

The historical sales tables consume the majority of memory because every day is stored as a separate column. During ETL, these wide tables will be transformed into a normalized daily transaction format for more efficient analytics and machine learning.

## Dataset Relationships

The M5 source files connect through several shared identifiers:

- `item_id` identifies individual products.
- `store_id` identifies individual stores.
- `wm_yr_wk` connects weekly sell prices to the calendar.
- `d` connects daily sales columns (`d_1`, `d_2`, ...) to actual calendar dates.
- `dept_id` and `cat_id` define the product hierarchy.
- `state_id` connects stores to their states.

These relationships are used later to transform the raw competition files into relational warehouse tables.

In [27]:
# Summary statistics for sell prices
prices["sell_price"].describe()

count    6.841121e+06
mean     4.410952e+00
std      3.408814e+00
min      1.000000e-02
25%      2.180000e+00
50%      3.470000e+00
75%      5.840000e+00
max      1.073200e+02
Name: sell_price, dtype: float64

In [28]:
# Summary statistics for historical daily sales
day_columns = [col for col in sales.columns if col.startswith("d_")]

sales[day_columns].stack().describe()

count    5.832737e+07
mean     1.126322e+00
std      3.873108e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      7.630000e+02
dtype: float64

## Summary Statistics

The summary statistics provide an initial view of product pricing and historical demand in the Walmart M5 dataset.

### Pricing

The sell price dataset contains approximately **6.84 million price records**.

- Average sell price: **$4.41**
- Median sell price: **$3.47**
- 25% of prices are below **$2.18**
- 75% of prices are below **$5.84**
- Prices range from **$0.01 to $107.32**

The difference between the median and maximum price indicates that most products are relatively low-priced, while a smaller number of products have substantially higher prices.

### Historical Demand

The historical sales data contains approximately **58.3 million daily product-store observations**.

- Average daily sales: **1.13 units**
- Median daily sales: **0 units**
- 75% of observations are **1 unit or less**
- Maximum observed daily sales: **763 units**

A median of zero indicates that many product-store combinations have days with no sales. At the same time, the maximum of 763 units shows that occasional large demand spikes occur.

### Key Takeaway

Demand is therefore highly uneven across products and time. Most product-store-day observations have low or zero sales, while occasional periods experience much higher demand.

These characteristics will be important later when creating lag features, rolling averages, seasonal features, and demand forecasting models.

## Key Findings

The initial data exploration identified several important characteristics of the Walmart M5 dataset:

- The dataset contains **3,049 products across 10 stores and 3 states**.
- Historical sales are recorded at the **daily product-store level**.
- Sales data is stored in a wide format and will require transformation before modeling.
- Product prices vary over time and are stored separately from daily sales.
- Calendar data provides dates, events, weekdays, and SNAP information that can help explain changes in demand.
- Most product-store-day observations have low or zero sales, while occasional large demand spikes occur.
- No unexpected duplicate records or missing values were identified in the primary sales and pricing data.

## Next Steps

With the structure and quality of the source data understood, the next stages of the project will:

1. Analyze historical sales, pricing, product, store, and seasonal patterns.
2. Transform the raw datasets into analysis-ready warehouse tables.
3. Engineer features such as lagged sales, rolling averages, pricing changes, holidays, weather, and economic indicators.
4. Build and evaluate demand forecasting models.
5. Use demand forecasts to support the dynamic pricing system.